In [ ]:
import pandas as pd
import os

def load_and_standardize(file_path):
    # 1. Deteksi ekstensi file untuk metode ekstraksi yang tepat
    _, ext = os.path.splitext(file_path)
    
    if ext.lower() in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    elif ext.lower() == '.csv':
        df = pd.read_csv(file_path, delimiter=';', encoding='utf-16')
    else:
        raise ValueError(f"Format file tidak dikenali: {ext}")

    # 2. Standarisasi Header
    df.columns = df.columns.str.strip()
    
    # ---------------------------------------------------------
    if ext.lower() in ['.xlsx', '.xls']:
        df.rename(columns={'Load KWH Refiner': 'Load KWH Tickling Refiner'}, inplace=True)
    # ---------------------------------------------------------

    rename_map = {
        'Yangkee  Speed': 'Yankee Speed',
        'Coating Flow': 'Flow Coating',
        'Release Flow': 'Flow Release',
        'Jet Rasio': 'Jet Wire Ratio',
        'Yangkee Temperatur' : 'Yankee Temperature',
        'Hood Tempetatur' : 'Hood Temperature',
        'Load Ampere Turbo Vakum' : 'Load Ampere Turbo Vacuum'
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
    
    # 3. Validasi Waktu sebagai Kunci Utama
    df.dropna(subset=['Time'], inplace=True) 
    df['Time'] = pd.to_datetime(df['Time'], dayfirst=True, format='mixed', errors='coerce')
    
    return df

def build_master_pipeline(file_list):
    dataframes = []
    
    for file in file_list:
        try:
            df = load_and_standardize(file)
            dataframes.append(df)
        except FileNotFoundError:
            print(f"Peringatan: File {file} tidak ditemukan. Dilewati.")
            
    if not dataframes:
        raise ValueError("Tidak ada data yang berhasil dimuat.")

    # 4. Penggabungan dan Pembersihan Data Redundan
    master_df = pd.concat(dataframes, ignore_index=True)
    master_df.drop_duplicates(subset=['Time'], keep='last', inplace=True)
    master_df.sort_values('Time', inplace=True)
    
    return master_df.reset_index(drop=True)

# Eksekusi Pipeline
file_sources = [
    'PM Params/Maret-April PM 15.xlsx',
    'PM Params/04052026_PM15.csv'
]

raw15 = build_master_pipeline(file_sources)
raw15.info()

<class 'pandas.DataFrame'>
RangeIndex: 86345 entries, 0 to 86344
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Time                       86345 non-null  datetime64[us]
 1   Yankee Speed               86345 non-null  float64       
 2   Pope Reel Speed            86345 non-null  float64       
 3   Yankee Pressure            86345 non-null  float64       
 4   Stock Flow                 86345 non-null  float64       
 5   Stock Consistency          86345 non-null  float64       
 6   Flow Coating               86345 non-null  float64       
 7   Flow Release               86345 non-null  float64       
 8   Jet Wire Ratio             86345 non-null  float64       
 9   Load KWH Tickling Refiner  86345 non-null  float64       
 10  Yankee Temperature         23160 non-null  float64       
 11  Load KWH Refiner           23160 non-null  float64       
 12  Hood Tempetatur